# 01 — Bốn biến thể

So sánh cùng một phép chia dữ liệu. Mọi transformer fit trên train, transform lại val/test.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experiment.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from heart_disease.config import load_settings
from heart_disease.data import load_cleveland, split_dataset
settings = load_settings()
raw = load_cleveland(settings.raw_path)
splits = split_dataset(raw, settings)


In [ ]:
import numpy as np
import pandas as pd
from heart_disease.pipelines import build_feature_pipeline

feature_pipelines, rows = {}, []
for variant in settings.variants:
    pipeline = build_feature_pipeline(variant, settings)
    pipeline.fit(*splits["train"])
    feature_pipelines[variant] = pipeline
    for name, (X, y) in splits.items():
        transformed = pipeline.transform(X)
        assert transformed.index.equals(y.index)
        assert np.isfinite(transformed.to_numpy()).all()
        rows.append(dict(variant=variant, split=name, rows=len(X), features=transformed.shape[1]))
display(pd.DataFrame(rows))


## Cột đã được chọn

MI xếp hạng từng feature; DT importance phụ thuộc cây đã fit. Điểm 0 không khẳng định feature vô ích cho mọi model.

In [ ]:
for variant, pipeline in feature_pipelines.items():
    print(variant, pipeline.get_feature_names_out().tolist())
    for selector in ("mi", "dt"):
        if selector in pipeline.named_steps:
            display(pipeline.named_steps[selector].scores_.rename("score").to_frame())


## Xuất dữ liệu

CSV dùng để kiểm tra/nộp. Khi tuning vẫn dùng pipeline đầy đủ trên dữ liệu chưa transform.

In [ ]:
from heart_disease.artifacts import export_variants
exported = export_variants(splits, settings, ROOT / "data/processed")
print(len(exported["csv_paths"]), exported["archive"])


## Nhận xét về đặc trưng đã hoàn thiện

Original sau ColumnTransformer có năm numeric trước, tám categorical sau, không giữ nguyên thứ tự CSV. Original và FE có 13 cột; hai biến thể DT có 10 cột. Original+DT bỏ trestbps/fbs/restecg khỏi Top-10 trên train cuối. FE giữ thal_3.0, thal_7.0, cp_4.0, ca_0.0 và các feature khác theo MI. exang_0/exang_1 có cùng MI vì hai mã bổ sung, không là hai nguồn thông tin độc lập. MI và DT đo khác nhau nên thứ hạng có thể lệch. Danh sách có thể thay đổi giữa fold; không chọn lại theo val/test. Bốn biến thể đồng thời thay nhiều bước nên chưa tách riêng tác động nhân quả của từng phép FE.